# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

# Projektpfade
RAW_PATH = Path("../data/raw/brazilian-ecommerce")
DB_PATH = Path("../data/processed/olist.duckdb")

# Verbindung zur DuckDB als Datei (persistente DB statt in-memory)
con = duckdb.connect(DB_PATH.as_posix())

# Hilfsfunktion für SQL-Abfragen
def sql(query: str):
    return con.execute(query).df()

# Alle CSV-Dateien in DuckDB als Tabellen speichern (persistent in olist.duckdb)
for file in RAW_PATH.glob("*.csv"):
    table_name = file.stem.replace("olist_", "").replace("_dataset", "")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{file.as_posix()}');
    """)

# Alle Tabellen anzeigen
sql("SHOW TABLES")

## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [ ]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
0,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,47813,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,118.70,22.76,1
1,9b18f3fc296990b97854e351334a32f6,b2cac0b16835dabf811b204127f58afa,carapicuiba,SP,06330,138849fd84dff2fb4ca70a0a34c4aa1c,delivered,2018-02-01 14:02:19,2018-02-03 02:53:07,39.47,13.37,1
2,bb2f5e670f7155dc622c57e4b31d0a69,31b8fa2573bde01af4737e8ed29c348b,sao paulo,SP,02346,a6aeb116d2cb5013eb8a94585b71ffef,delivered,2017-09-13 14:27:11,2017-09-13 14:44:39,50.00,9.34,1
3,f26a435864aebedff7f7c84f82ee229f,bb4d84a2b45b22ed710ac8c0dec63d1a,poa,SP,08552,b8801cccd8068de30112e4f49903d74a,delivered,2017-07-30 03:06:35,2017-07-30 03:25:08,19.99,7.78,1
4,803ac05904124294f8767894d6da532b,34c58672601f2c6d29db7efd1f6bf958,bonfinopolis de minas,MG,38650,bfe42c22ecbf90bc9f35cf591270b6a7,delivered,2018-01-27 22:04:34,2018-01-27 22:16:18,27.30,15.10,1
...,...,...,...,...,...,...,...,...,...,...,...,...
101565,0b2bbe8811e7f1cb2b3e25c95d276116,33a76918feef5cb88ff4b0b16f1611b4,paranavai,PR,87711,521e93507df5620aaee015ba441ec756,delivered,2017-09-09 18:40:40,2017-09-09 18:55:08,199.90,24.22,1
101566,8f8218f15015d63e71aea6a68d9e1b67,94b9a50e7c20bb52bb4e7d4f174fc3b6,upanema,RN,59670,d40516d201a6c180696f31d77ca651ac,delivered,2018-03-13 21:58:21,2018-03-13 22:40:24,39.90,9.23,1
101567,75c8873878785af64664ea575dc50c52,7e53420f2126366b3489c18314f82803,uberlandia,MG,38414,db6c6df011e1bcc8e03e81b23982a1dc,delivered,2017-05-11 12:10:03,2017-05-11 13:05:31,68.90,17.49,1
101568,0a7cdb607bacd00cb014013d91f5a218,695cb366a3030919b19a244401a8f017,osasco,SP,06070,74414263bc2db93856b800f04ded2cab,delivered,2018-04-12 16:49:57,2018-04-13 13:29:53,39.90,11.37,1


In [ ]:
df_rfm_eda.describe()

#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [ ]:
df_rfm_eda.dtypes

In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

In [ ]:
df_rfm_eda.describe()

In [ ]:
df_rfm_eda.isna().sum()

In [ ]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10385,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,59.900002,17.160000,1
22909,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,135.000000,19.230000,1
31789,4c1ccc74e00993733742a3c786dc3c1f,91efb7fcabc17925099dced52435837f,novo hamburgo,RS,93548,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,379.000000,17.860001,1
39443,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,79.989998,26.820000,1
41718,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,79.989998,15.770000,1
50891,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,309.899994,39.110001,1
51220,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,39.990002,14.520000,1
51852,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,49.000000,14.520000,2
57734,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,149.800003,13.630000,1
63729,29c35fc91fc13fb5073c8f30505d860d,7e1a5ca61b572d76b64b6688b9f96473,caninde,CE,62700,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,79.989998,26.820000,1


In [ ]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

In [ ]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

In [ ]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

In [ ]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


In [ ]:
test['sum_status']=test.sum(axis=1)

In [ ]:
test.loc[test['sum_status']!=1, :] 

####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [ ]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [ ]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

## Filterung nach der Bestellung mit den meisten Duplikaten

In [ ]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10700,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,309.000000,1.84,1
11883,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,56.000000,3.68,2
15379,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,159.000000,3.67,2
26454,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,95.900002,0.15,2
42659,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,63.700001,0.15,1
75796,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,33.900002,1.84,1
99696,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,109.900002,0.15,1


In [ ]:
df_rfm_eda.describe()

### EDA für zweite Kernaufgabe

In [20]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [21]:
df_pc_eda

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
0,fd25ab760bfbba13c198fa3b4f1a0cd3,sports_leisure,2,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,185.00,13.63,4
1,154e7e31ebfa092203795c972e5804a6,health_beauty,1,delivered,2017-10-02 11:17:07,2017-10-02 11:28:29,23.99,14.10,4
2,d0fe4295267f15ccaceac4fb233d8c9a,computers_accessories,1,delivered,2018-04-07 15:24:30,2018-04-07 15:35:10,45.90,8.82,5
3,5cbd407f3315b628a89206fbc140f6c8,market_place,1,delivered,2018-03-28 16:16:20,2018-03-30 03:08:56,19.90,7.39,5
4,a0a6b0afd47416d62bb25892c68b6296,garden_tools,1,delivered,2017-12-29 12:29:00,2017-12-29 12:46:44,62.40,15.88,5
...,...,...,...,...,...,...,...,...,...
100703,eebbed5ed3b134eceb717496c47652ba,bed_bath_table,1,delivered,2017-08-19 20:25:59,2017-08-22 04:05:17,99.99,48.23,<NA>
100704,16b691e994cb81b2e7e31c93ba603136,bed_bath_table,1,delivered,2018-05-11 18:52:04,2018-05-15 04:15:26,58.99,17.32,<NA>
100705,ba3a1e2c6cc1fb7a27dd74916212e6fb,toys,1,delivered,2018-01-24 23:12:41,2018-01-24 23:32:51,29.90,14.10,<NA>
100706,741a31499a578979be85db7f80139e62,furniture_decor,1,delivered,2017-12-14 00:06:50,2017-12-14 00:17:38,45.90,16.11,<NA>


In [22]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
count,100708.000000,100708,100695,100708.000000,100708.000000,99940.0
mean,1.103597,2018-01-01 16:12:59.141210,2018-01-02 03:32:58.710333,124.151016,20.142170,4.088883
min,1.000000,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.0
25%,1.000000,2017-09-13 17:09:08,2017-09-14 02:45:40,40.140000,13.180000,4.0
50%,1.000000,2018-01-20 13:59:55.500000,2018-01-20 20:00:10,78.000000,16.360000,5.0
75%,1.000000,2018-05-05 21:22:12.750000,2018-05-06 13:50:11,139.000000,21.260000,5.0
max,20.000000,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,5.0
std,0.461837,NaN,NaN,187.484910,15.898289,1.342309


In [23]:
df_pc_eda.dtypes

product_id                               object
product_category_name_english            object
order_count                               int64
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
price                                   float64
freight_value                           float64
review_score                              Int64
dtype: object

In [24]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

product_id                            category
product_category_name_english         category
order_count                              Int16
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
price                                  float32
freight_value                          float32
review_score                          category
dtype: object

In [25]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value
count,100708.0,100708,100695,100708.000000,100708.000000
mean,1.103597,2018-01-01 16:12:59,2018-01-02 03:32:58,124.151009,20.142170
min,1.0,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000
25%,1.0,2017-09-13 17:09:08,2017-09-14 02:45:40,40.139999,13.180000
50%,1.0,2018-01-20 13:59:55,2018-01-20 20:00:10,78.000000,16.360001
75%,1.0,2018-05-05 21:22:12,2018-05-06 13:50:11,139.000000,21.260000
max,20.0,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993
std,0.461837,NaN,NaN,187.484909,15.898289


In [26]:
df_pc_eda.isna().sum()

product_id                         0
product_category_name_english      0
order_count                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 13
price                              0
freight_value                      0
review_score                     768
dtype: int64

In [27]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
1552,6f3b5b605d91b7439c5e3f5a8dffeea7,watches_gifts,1,delivered,2018-03-19 14:28:12,2018-03-19 14:49:35,165.000000,19.030001,NaN
1553,154b6772d166a9c3398661ea26350dc6,sports_leisure,1,delivered,2018-03-07 12:27:19,2018-03-09 18:29:23,399.899994,37.930000,NaN
1554,ebc714d9f070a89f4d411982ea9670fb,toys,1,delivered,2017-12-06 10:51:17,2017-12-06 11:14:49,499.989990,23.990000,NaN
1555,165f86fe8b799a708a20ee4ba125c289,cool_stuff,1,delivered,2018-04-12 15:15:43,2018-04-12 15:30:49,169.990005,15.180000,NaN
1556,fb55982be901439613a95940feefd9ee,stationery,1,delivered,2017-12-20 21:44:31,2017-12-20 21:56:28,79.000000,13.570000,NaN
1557,1fa0faff5eafb13003f9559ebe6becb3,watches_gifts,1,delivered,2018-03-16 19:39:14,2018-03-16 19:55:22,59.000000,15.290000,NaN
1558,d6fe3b4ddecd4a8393c6a1385de3bfb6,office_furniture,3,delivered,2017-03-09 23:42:58,2017-03-09 23:42:58,199.990005,34.439999,NaN
1559,a92930c327948861c015c919a0bcb4a8,watches_gifts,1,delivered,2017-06-19 01:46:34,2017-06-20 11:35:25,78.000000,7.800000,NaN
1560,3d77287739b6bf1ac163ce2d77570ada,furniture_decor,2,delivered,2017-04-21 00:11:35,2017-04-21 01:05:19,449.899994,124.540001,NaN
1561,e0f55a53bebaae74a51a8fb9639681d6,baby,2,delivered,2018-05-04 09:49:02,2018-05-04 10:11:27,18.400000,12.790000,NaN


In [28]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

### EDA für dritte Kernaufgabe

In [29]:
# Dataframe für Aufgabe 3. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
        r.review_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp,
    oi.product_id,
    pcnt.product_category_name_english,
    Count(oi.product_id) AS product_count,                
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,              
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN geolocation s_geo ON s.seller_zip_code_prefix = s_geo.geolocation_zip_code_prefix
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
WHERE o.order_status IN ('delivered')
Group BY o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
         o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
         r.review_id, r.review_score, r.review_comment_title, r.review_comment_message, 
                     r.review_creation_date, r.review_answer_timestamp,
         oi.product_id, s.seller_id, s.seller_city, s.seller_state, c.customer_unique_id,
                     c.customer_city, c.customer_state, pcnt.product_category_name_english
    """)
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_unique_id,customer_city,customer_state
0,2bca9fb7f37de0ca85a20234cdce1fb1,delivered,2018-01-06 09:54:02,2018-01-09 07:19:00,2018-01-11 17:25:05,2018-02-02 01:38:41,2018-02-09,c71468ac79d7dd35d94fdb8082dd14d3,5,None,...,2018-02-03 02:40:53,51d646c5c93e0f1de543528d0e24eadc,health_beauty,94,c35672b10ad50968f567ea3f4b91e877,mesquita,RJ,aa200cd9deaa201aa2624b22425c8c38,cuiaba,MT
1,c0aa7b8e7596f25711073d95e0dbbec5,delivered,2018-08-18 13:20:33,2018-08-18 13:35:12,2018-08-20 14:33:00,2018-08-23 21:19:32,2018-09-11,378172204c8f0ed00dc2d34fcb89b720,5,None,...,2018-08-27 23:09:15,29af47aaefb3a7a8bc5b00d008cdba89,watches_gifts,14,056b4ada5bbc2c50cc7842547dda6b51,queimados,RJ,581499ad4abed228b0791d946e982a50,contagem,MG
2,e6ffcd88cc3d2795303c2fd25cadf74a,delivered,2018-05-06 12:35:54,2018-05-06 12:53:22,2018-05-09 08:46:00,2018-05-28 20:06:38,2018-06-01,2c09968c6547b21908c356ba0894d8c3,5,None,...,2018-05-29 23:14:34,51d646c5c93e0f1de543528d0e24eadc,health_beauty,94,c35672b10ad50968f567ea3f4b91e877,mesquita,RJ,bc0cc7fd03864119501b931413dde0df,jaboatao dos guararapes,PE
3,c4c1257d437761ebf697da82b5cca4e1,delivered,2018-07-23 20:53:38,2018-07-24 10:31:48,2018-07-25 13:54:00,2018-07-31 22:51:34,2018-08-13,d423f225b861a8ee0ab77ae4aae73e88,5,None,...,2018-08-03 18:56:23,2ff995aead9c63a1f37a07b3664ead37,furniture_living_room,69,8b9d6eec4a7eb7d0f9d579ce0b38324d,mesquita,RJ,4d9e104764077f7dfae917c7cc803212,vila velha,ES
4,f661f51c758806f369cc3c485ff6e936,delivered,2018-07-17 19:00:52,2018-07-17 19:21:50,2018-07-19 12:50:00,2018-07-21 17:44:32,2018-07-26,ff27cba30c9821de2f6f97c84c838cec,5,None,...,2018-07-28 15:42:26,2ff995aead9c63a1f37a07b3664ead37,furniture_living_room,69,8b9d6eec4a7eb7d0f9d579ce0b38324d,mesquita,RJ,38c5c1807e2d52f9d89289b5dedb4bda,teresopolis,RJ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98435,dc5e5ca2d07c93091a87b58d9888769a,delivered,2018-05-02 11:04:44,2018-05-02 11:15:18,2018-05-02 13:17:00,2018-05-08 17:44:30,2018-05-21,968ee2cc13a5ddbd90dde0e7a7392047,5,None,...,2018-05-16 18:39:06,2ef440beadfeaa9ac762aaf8fead2836,pet_shop,132,232a6014e7b10cba61c6c2b2ea6bb4b0,cafelandia,SP,a045fa902e700cb21b00a496d51ef016,serrana,SP
98436,97e22a7b438827d5a86bc1a9f92d8c83,delivered,2018-01-12 20:00:28,2018-01-12 20:08:25,2018-01-15 18:27:01,2018-01-31 19:13:06,2018-02-16,cba1bfb26ce12243350855b93b57dcce,4,None,...,2018-02-02 14:42:38,1a69b9ec6d25d7656d024ad51f55c0ec,bed_bath_table,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,203be24e6117b285f1a0e57eab8d4b41,caxias do sul,RS
98437,1e575fc9d2c56fa7889923cd298830f1,delivered,2018-07-21 22:33:25,2018-07-21 22:45:12,2018-07-25 06:09:00,2018-07-27 13:58:11,2018-08-13,1f372c36cacaba1245f96b3884f07e86,5,None,...,2018-07-29 18:14:03,286bcf62e47dfc1e51cf0d525bd4d1bf,bed_bath_table,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,0697fbaa53cc4ce49707782afb2c17a9,guaruja,SP
98438,8651a94589a7baddf5213b1703c98ad6,delivered,2018-06-19 11:59:59,2018-06-19 12:18:30,2018-06-20 14:13:00,2018-07-20 16:24:48,2018-07-24,6298846cdf0eceb7ba5661c3810725b7,1,Atrasado e com defeito,...,2018-07-23 22:45:15,1938ab47ef011dae9e4ced458166432e,baby,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,f226b323a26dc6e66d61c7d93d05ae4b,serrinha,BA


In [30]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98440,98427,98438,98432,98440,98440.000000,98440,98440,98440.000000
mean,2018-01-02 12:41:22.360717,2018-01-03 00:00:31.715667,2018-01-05 17:51:58.231882,2018-01-14 23:24:48.813424,2018-01-26 07:44:39.008533,4.127611,2018-01-14 18:11:59.122308,2018-01-17 21:45:12.709285,159.316457
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,2016-10-06 00:00:00,2016-10-07 18:32:28,1.000000
25%,2017-09-14 14:05:40.500000,2017-09-14 22:27:59,2017-09-18 19:27:43.250000,2017-09-26 16:50:49.250000,2017-10-05 00:00:00,4.000000,2017-09-27 00:00:00,2017-09-29 13:07:05.750000,58.000000
50%,2018-01-21 14:55:19.500000,2018-01-22 14:11:57,2018-01-24 19:22:26,2018-02-02 22:53:02,2018-02-16 00:00:00,5.000000,2018-02-03 00:00:00,2018-02-06 22:17:57,117.000000
75%,2018-05-06 19:52:50.500000,2018-05-07 17:17:32.500000,2018-05-09 10:55:15,2018-05-16 18:05:47.500000,2018-05-28 00:00:00,5.000000,2018-05-17 00:00:00,2018-05-20 20:30:33.250000,225.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35,4638.000000
std,NaN,NaN,NaN,NaN,NaN,1.308952,NaN,NaN,155.447125


In [31]:
df_service_eda.dtypes


order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_id                                object
review_score                              int64
review_comment_title                     object
review_comment_message                   object
review_creation_date             datetime64[us]
review_answer_timestamp          datetime64[us]
product_id                               object
product_category_name_english            object
product_count                             int64
seller_id                                object
seller_city                              object
seller_state                             object
customer_unique_id                       object
customer_city                           

In [32]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]',
              'review_id': 'category',
              'review_score': 'Int16',
              'review_comment_title': 'category',
              'review_comment_message': 'category',
              'review_creation_date': 'datetime64[s]',
              'review_answer_timestamp': 'datetime64[s]',
              'product_id': 'category',
              'product_category_name_english': 'category',
              'product_count': 'Int16',
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_unique_id': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

order_id                              category
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
order_delivered_carrier_date     datetime64[s]
order_delivered_customer_date    datetime64[s]
order_estimated_delivery_date    datetime64[s]
review_id                             category
review_score                             Int16
review_comment_title                  category
review_comment_message                category
review_creation_date             datetime64[s]
review_answer_timestamp          datetime64[s]
product_id                            category
product_category_name_english         category
product_count                            Int16
seller_id                             category
seller_city                           category
seller_state                          category
customer_unique_id                    category
customer_city                         category
customer_stat

In [33]:
df_service_eda.isna().sum()

order_id                             0
order_status                         0
order_purchase_timestamp             0
order_approved_at                   13
order_delivered_carrier_date         2
order_delivered_customer_date        8
order_estimated_delivery_date        0
review_id                            0
review_score                         0
review_comment_title             86754
review_comment_message           57956
review_creation_date                 0
review_answer_timestamp              0
product_id                           0
product_category_name_english        0
product_count                        0
seller_id                            0
seller_city                          0
seller_state                         0
customer_unique_id                   0
customer_city                        0
customer_state                       0
dtype: int64

In [34]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_unique_id,customer_city,customer_state
15641,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,f48c6c944a5d52dcca8ac5c4ec417cf2,5,NaN,...,2017-12-19 04:15:39,a50acd33ba7a8da8e9db65094fa990a4,auto,30,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,13467e882eb3a701826435ee4424f2bd,cerquilho,SP
19117,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,ee2d30652e2f7fc00861074f795f5bf0,5,Excelente!,...,2018-07-07 18:48:09,ec165cd31c50585786ffda6feff5d0a6,toys,59,8bdd8e3fd58bafa48af76b2c5fd71974,sao paulo,SP,ebf7e0d43a78c81991a4c59c145c75db,sao carlos,SP
24283,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,0d4c56af896dd6eb9de8edbaa1902d22,1,Péssimo,...,2018-06-16 13:55:00,a2a7efc985315e86d4f0f705701b342b,computers_accessories,171,ed4acab38528488b65a9a9c603ff024a,sao paulo,SP,cce5e8188bf42ffb3bb5b18ff58f5965,guarulhos,SP
29147,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14,e945d1831a3d98008913fc31dcbb804d,5,NaN,...,2017-10-17 10:56:02,44c2baf621113fa7ac95fa06b4afbc68,furniture_decor,57,3f2af2670e104d1bcb54022274daeac5,terra boa,PR,a2ac81ecc3704410ae240e74d4f0af40,indaiatuba,SP
45086,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,c0dd6bec0375c376f044af102118526f,5,Entrega super rápida.,...,2018-06-29 16:26:37,2167c8f6252667c0eb9edd51520706a1,industry_commerce_and_business,176,0bb738e4d789e63e2267697c42d35a2d,sao roque,SP,2f17c5b324ad603491521b279a9ff4de,quadra,SP
59750,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,d055795a562efffefe47ef81e5435322,5,Muito bom,...,2018-07-06 20:30:17,55bfa0307d7a46bed72c492259921231,books_general_interest,90,343e716476e3748b069f980efbaa294e,campinas,SP,175378436e2978be55b8f4316bce4811,ribeirao pires,SP
66575,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,bb311d9562ecbefc8e4be756d8999892,5,NaN,...,2018-07-10 11:38:13,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,19,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,3bc508d482a402715be4d5cf4020cc81,sumare,SP
66583,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,25e11638a3d01a87e8e62338a39eee28,5,NaN,...,2018-07-11 19:27:46,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,19,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,1bd06a0c0df8b23dacfd3725d2dc0bb9,pindamonhangaba,SP
91532,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,4e755f114e50d33b9ac6a56e0d7d3ea9,5,NaN,...,2017-06-27 01:49:04,30b5b5635a79548a48d04162d971848f,sports_leisure,67,f9bbdd976532d50b7816d285a22bd01e,sao paulo,SP,d77cf4be2654aa70ef150f8bfec076a6,porto alegre,RS


In [35]:
df_service_eda['order_delivered_customer_date'] = df_service_eda['order_delivered_customer_date'].fillna(
    pd.to_datetime(df_service_eda['order_estimated_delivery_date'])
)

df_service_eda['order_delivered_carrier_date'] = df_service_eda['order_delivered_carrier_date'].fillna(
    pd.to_datetime(df_service_eda['order_approved_at']) + pd.Timedelta(days=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_unique_id,customer_city,customer_state


In [36]:
df_service_eda.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98440,98440,98440,98440,98440,98440.0,98440,98440,98440.0
mean,2018-01-02 12:41:22,2018-01-02 22:59:04,2018-01-05 17:47:23,2018-01-14 23:37:38,2018-01-26 07:44:39,4.127611,2018-01-14 18:11:59,2018-01-17 21:45:12,159.316457
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.0,2016-10-06 00:00:00,2016-10-07 18:32:28,1.0
25%,2017-09-14 14:05:40,2017-09-14 21:49:00,2017-09-18 19:27:41,2017-09-26 16:54:58,2017-10-05 00:00:00,4.0,2017-09-27 00:00:00,2017-09-29 13:07:05,58.0
50%,2018-01-21 14:55:19,2018-01-22 14:04:59,2018-01-24 19:20:35,2018-02-02 23:00:57,2018-02-16 00:00:00,5.0,2018-02-03 00:00:00,2018-02-06 22:17:57,117.0
75%,2018-05-06 19:52:50,2018-05-07 17:17:17,2018-05-09 10:49:45,2018-05-16 18:11:39,2018-05-28 00:00:00,5.0,2018-05-17 00:00:00,2018-05-20 20:30:33,225.0
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.0,2018-08-31 00:00:00,2018-10-29 12:27:35,4638.0
std,NaN,NaN,NaN,NaN,NaN,1.308952,NaN,NaN,155.447125


In [37]:
print("Gesamte Duplikate:", df_service_eda.duplicated().sum())


In [38]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_service_eda[(df_service_eda['order_id'] == order_id)]

order_data.head(63).sort_values('order_purchase_timestamp', ascending=True)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_unique_id,customer_city,customer_state
45708,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,5ddab10d5e0a23acb99acf56b62b3276,housewares,57,3d0cd21d41671c46f82cd11176bf7277,joinville,SC,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP
65797,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,ebf9bc6cd600eadd681384e3116fda85,bed_bath_table,46,822166ed1e47908f7cfb49946d03c726,tres rios,RJ,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP


In [39]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_unique_id,customer_city,customer_state
0,2bca9fb7f37de0ca85a20234cdce1fb1,delivered,2018-01-06 09:54:02,2018-01-09 07:19:00,2018-01-11 17:25:05,2018-02-02 01:38:41,2018-02-09,c71468ac79d7dd35d94fdb8082dd14d3,5,NaN,...,2018-02-03 02:40:53,51d646c5c93e0f1de543528d0e24eadc,health_beauty,94,c35672b10ad50968f567ea3f4b91e877,mesquita,RJ,aa200cd9deaa201aa2624b22425c8c38,cuiaba,MT
1,c0aa7b8e7596f25711073d95e0dbbec5,delivered,2018-08-18 13:20:33,2018-08-18 13:35:12,2018-08-20 14:33:00,2018-08-23 21:19:32,2018-09-11,378172204c8f0ed00dc2d34fcb89b720,5,NaN,...,2018-08-27 23:09:15,29af47aaefb3a7a8bc5b00d008cdba89,watches_gifts,14,056b4ada5bbc2c50cc7842547dda6b51,queimados,RJ,581499ad4abed228b0791d946e982a50,contagem,MG
2,e6ffcd88cc3d2795303c2fd25cadf74a,delivered,2018-05-06 12:35:54,2018-05-06 12:53:22,2018-05-09 08:46:00,2018-05-28 20:06:38,2018-06-01,2c09968c6547b21908c356ba0894d8c3,5,NaN,...,2018-05-29 23:14:34,51d646c5c93e0f1de543528d0e24eadc,health_beauty,94,c35672b10ad50968f567ea3f4b91e877,mesquita,RJ,bc0cc7fd03864119501b931413dde0df,jaboatao dos guararapes,PE
3,c4c1257d437761ebf697da82b5cca4e1,delivered,2018-07-23 20:53:38,2018-07-24 10:31:48,2018-07-25 13:54:00,2018-07-31 22:51:34,2018-08-13,d423f225b861a8ee0ab77ae4aae73e88,5,NaN,...,2018-08-03 18:56:23,2ff995aead9c63a1f37a07b3664ead37,furniture_living_room,69,8b9d6eec4a7eb7d0f9d579ce0b38324d,mesquita,RJ,4d9e104764077f7dfae917c7cc803212,vila velha,ES
4,f661f51c758806f369cc3c485ff6e936,delivered,2018-07-17 19:00:52,2018-07-17 19:21:50,2018-07-19 12:50:00,2018-07-21 17:44:32,2018-07-26,ff27cba30c9821de2f6f97c84c838cec,5,NaN,...,2018-07-28 15:42:26,2ff995aead9c63a1f37a07b3664ead37,furniture_living_room,69,8b9d6eec4a7eb7d0f9d579ce0b38324d,mesquita,RJ,38c5c1807e2d52f9d89289b5dedb4bda,teresopolis,RJ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98435,dc5e5ca2d07c93091a87b58d9888769a,delivered,2018-05-02 11:04:44,2018-05-02 11:15:18,2018-05-02 13:17:00,2018-05-08 17:44:30,2018-05-21,968ee2cc13a5ddbd90dde0e7a7392047,5,NaN,...,2018-05-16 18:39:06,2ef440beadfeaa9ac762aaf8fead2836,pet_shop,132,232a6014e7b10cba61c6c2b2ea6bb4b0,cafelandia,SP,a045fa902e700cb21b00a496d51ef016,serrana,SP
98436,97e22a7b438827d5a86bc1a9f92d8c83,delivered,2018-01-12 20:00:28,2018-01-12 20:08:25,2018-01-15 18:27:01,2018-01-31 19:13:06,2018-02-16,cba1bfb26ce12243350855b93b57dcce,4,NaN,...,2018-02-02 14:42:38,1a69b9ec6d25d7656d024ad51f55c0ec,bed_bath_table,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,203be24e6117b285f1a0e57eab8d4b41,caxias do sul,RS
98437,1e575fc9d2c56fa7889923cd298830f1,delivered,2018-07-21 22:33:25,2018-07-21 22:45:12,2018-07-25 06:09:00,2018-07-27 13:58:11,2018-08-13,1f372c36cacaba1245f96b3884f07e86,5,NaN,...,2018-07-29 18:14:03,286bcf62e47dfc1e51cf0d525bd4d1bf,bed_bath_table,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,0697fbaa53cc4ce49707782afb2c17a9,guaruja,SP
98438,8651a94589a7baddf5213b1703c98ad6,delivered,2018-06-19 11:59:59,2018-06-19 12:18:30,2018-06-20 14:13:00,2018-07-20 16:24:48,2018-07-24,6298846cdf0eceb7ba5661c3810725b7,1,Atrasado e com defeito,...,2018-07-23 22:45:15,1938ab47ef011dae9e4ced458166432e,baby,173,9e6229250fedbe05838fef417b74e7fb,mirandopolis,SP,f226b323a26dc6e66d61c7d93d05ae4b,serrinha,BA


In [40]:
df_geolocation= sql("""
SELECT
   geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
FROM geolocation
    """)
df_geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,01037,-23.545621,-46.639292
1,01046,-23.546081,-46.644820
2,01046,-23.546129,-46.642951
3,01041,-23.544392,-46.639499
4,01035,-23.541578,-46.641607
...,...,...,...
1000158,99950,-28.068639,-52.010705
1000159,99900,-27.877125,-52.224882
1000160,99950,-28.071855,-52.014716
1000161,99980,-28.388932,-51.846871


In [41]:
df_geolocation.describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [42]:
df_geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
dtype: int64

In [43]:
print("Gesamte Duplikate:", df_geolocation.duplicated().sum())

Gesamte Duplikate: 280009


In [44]:
# 1. Duplikate pro Spalte einzeln
print("📊 Duplikate pro Spalte:")
for col in df_geolocation.columns:
    dups = df_geolocation.duplicated(subset=[col], keep=False).sum()
    print(f"{col:25}: {dups:,} ({dups/len(df_geolocation)*100:.1f}%)")

# 2. Multi-Duplikate (2+ Spalten identisch)
print("\n🔗 Kombi-Duplikate:")
for i in range(2, 4):  # 2 bis 5 Spalten
    cols = df_geolocation.columns[:i]
    dups = df_geolocation.duplicated(subset=cols, keep=False).sum()
    print(f"{i} Spalten: {dups:,}")

# 3. Gesamte (alle Spalten)
print(f"\nPerfekte Duplikate: {df_geolocation.duplicated().sum()}")

# 4. Top Duplikat-Kombinationen
print("\n🏆 Häufigste Duplikat-Gruppen:")
dup_groups = df_geolocation.groupby([df_geolocation.columns[0], df_geolocation.columns[1]]).size()
dup_groups = dup_groups[dup_groups > 1].sort_values(ascending=False)
print(dup_groups.head(10))

📊 Duplikate pro Spalte:
geolocation_zip_code_prefix: 999,120 (99.9%)
geolocation_lat          : 415,477 (41.5%)
geolocation_lng          : 415,133 (41.5%)

🔗 Kombi-Duplikate:
2 Spalten: 412,920
3 Spalten: 411,716

Perfekte Duplikate: 280009

🏆 Häufigste Duplikat-Gruppen:
geolocation_zip_code_prefix  geolocation_lat
88220                        -27.102099         314
06414                        -23.495901         189
05145                        -23.506049         141
06414                        -23.490618         127
22620                        -23.005514         102
22640                        -23.004582          89
22775                        -22.965906          89
71936                        -15.841451          85
03015                        -23.537186          83
09781                        -23.727641          81
dtype: int64


In [45]:
# con.execute("CREATE TABLE customer_rfm AS SELECT * FROM df_rfm_eda")
# con.execute("CREATE TABLE product_category AS SELECT * FROM df_pc_eda")
# con.execute("CREATE TABLE service_analyse AS SELECT * FROM df_service_eda")


In [46]:
sql("SHOW TABLES")

,name
0,customer_rfm
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,product_category
8,product_category_name_translation
9,products


In [47]:
con.close()